# Understanding Unicode

In [ ]:
# a) What Unicode character does chr(0) return?
chr(0)

In [ ]:
# b) How does this character’s string representation (__repr__()) differ from its printed representation?
print(chr(0))

In [ ]:
# c) What happens when this character occurs in text? It may be helpful to play around with the 
# following in your Python interpreter and see if it matches your expectations:
"this is a test" + chr(0) + "string"
print("this is a test" + chr(0) + "string")

# Unicode Encodings

In [ ]:
# a) What are some reasons to prefer training our tokenizer on UTF-8 encoded bytes, rather than
# UTF-16 or UTF-32? It may be helpful to compare the output of these encodings for various
# input strings.

print("UTF-8 is better for ASCII char training for comparing to UTF-16 and UTF-32, it prevents heading zeros, hence offers more efficient merging for BPE.")

In [ ]:
# b) Consider the following (incorrect) function, which is intended to decode a UTF-8 byte string
# into a Unicode string. Why is this function incorrect? Provide an example of an input byte
# string that yields incorrect results.

def decode_utf8_bytes_to_str_wrong(bytestring: bytes):
    print(bytestring)
    return "".join([bytes([b]).decode("utf-8") for b in bytestring])

# function decodes bytes one by one, when there are multi-bytes char it would fail at first leading byte

a = '😀'.encode('utf-8')
decode_utf8_bytes_to_str_wrong(a)

In [ ]:
# c) Give a two-byte sequence that does not decode to any Unicode character(s).
print(b'\x53'.decode("utf-8"))
print(b'\xFE\xFE'.decode("utf-8"))

In [ ]:
# Input
corpus = """low low low low low
lower lower widest widest widest
newest newest newest newest newest newest"""

# Strings represent tokens here for readability.
# An implementation would use bytes instead.

# Pre-tokenization
token_freq = {
    ("l", "o", "w"): 5,
    ("l", "o", "w", "e", "r"): 2,
    ("w", "i", "d", "e", "s", "t"): 3,
    ("n", "e", "w", "e", "s", "t"): 6,
}

# Merge 1
pair_count = {
    ("l", "o"): 7, ("o", "w"): 7,
    ("w", "e"): 8, ("e", "r"): 2,
    ("w", "i"): 3, ("i", "d"): 3,
    ("d", "e"): 3, ("e", "s"): 9,
    ("s", "t"): 9, ("n", "e"): 6,
    ("e", "w"): 6,
}

# Highest count: 9. Tie: ("s", "t") > ("e", "s").
# Merge ("s", "t") → "st".
token_freq = {
    ("l", "o", "w"): 5,
    ("l", "o", "w", "e", "r"): 2,
    ("w", "i", "d", "e", "st"): 3,
    ("n", "e", "w", "e", "st"): 6,
}

# Merge 2
pair_count = {
    ("l", "o"): 7, ("o", "w"): 7,
    ("w", "e"): 8, ("e", "r"): 2,
    ("w", "i"): 3, ("i", "d"): 3,
    ("d", "e"): 3, ("e", "st"): 9,
    ("n", "e"): 6, ("e", "w"): 6,
}

# Highest count: 9.
# Merge ("e", "st") → "est".
token_freq = {
    ("l", "o", "w"): 5,
    ("l", "o", "w", "e", "r"): 2,
    ("w", "i", "d", "est"): 3,
    ("n", "e", "w", "est"): 6,
}

# Merge 3
pair_count = {
    ("l", "o"): 7, ("o", "w"): 7,
    ("w", "e"): 2, ("e", "r"): 2,
    ("w", "i"): 3, ("i", "d"): 3,
    ("d", "est"): 3, ("n", "e"): 6,
    ("e", "w"): 6, ("w", "est"): 6,
}

# Highest count: 7. Tie: ("o", "w") > ("l", "o").
# Merge ("o", "w") → "ow".
token_freq = {
    ("l", "ow"): 5,
    ("l", "ow", "e", "r"): 2,
    ("w", "i", "d", "est"): 3,
    ("n", "e", "w", "est"): 6,
}

# Merge 4
pair_count = {
    ("l", "ow"): 7, ("ow", "e"): 2,
    ("e", "r"): 2, ("w", "i"): 3,
    ("i", "d"): 3, ("d", "est"): 3,
    ("n", "e"): 6, ("e", "w"): 6,
    ("w", "est"): 6,
}

# Highest count: 7.
# Merge ("l", "ow") → "low".
token_freq = {
    ("low",): 5,
    ("low", "e", "r"): 2,
    ("w", "i", "d", "est"): 3,
    ("n", "e", "w", "est"): 6,
}

# Merge 5
pair_count = {
    ("low", "e"): 2, ("e", "r"): 2,
    ("w", "i"): 3, ("i", "d"): 3,
    ("d", "est"): 3, ("n", "e"): 6,
    ("e", "w"): 6, ("w", "est"): 6,
}

# Highest count: 6.
# Tie: ("w", "est") > ("n", "e") > ("e", "w").
# Merge ("w", "est") → "west".
token_freq = {
    ("low",): 5,
    ("low", "e", "r"): 2,
    ("w", "i", "d", "est"): 3,
    ("n", "e", "west"): 6,
}

# Merge 6
pair_count = {
    ("low", "e"): 2, ("e", "r"): 2,
    ("w", "i"): 3, ("i", "d"): 3,
    ("d", "est"): 3, ("n", "e"): 6,
    ("e", "west"): 6,
}

# Highest count: 6. Tie: ("n", "e") > ("e", "west").
# Merge ("n", "e") → "ne".
token_freq = {
    ("low",): 5,
    ("low", "e", "r"): 2,
    ("w", "i", "d", "est"): 3,
    ("ne", "west"): 6,
}

# Ordered merges
merges = [
    ("s", "t"),
    ("e", "st"),
    ("o", "w"),
    ("l", "ow"),
    ("w", "est"),
    ("n", "e"),
]

# Final vocabulary:
# ["<|endoftext|>"] + all 256 byte tokens
#                  + ["st", "est", "ow", "low", "west", "ne"]

# "newest" tokenizes as ["ne", "west"].